# Slow Queries

Este cuaderno ayuda a diferenciar volumen de consultas lentas, severidad y detalle de outliers.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_CWD = Path.cwd()
candidate_roots = [
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == 'notebooks' else NOTEBOOK_CWD,
    NOTEBOOK_CWD / 'observability' / 'd365-fo-observability',
]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'kql_runner.py').exists() and (candidate / 'queries').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('No se pudo localizar observability/d365-fo-observability desde el directorio actual')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kql_runner import build_client, load_config, load_kql, metric_snapshot, plot_bar, plot_timeseries, run_kql, set_analyst_theme

config = load_config()
client = build_client(config=config)
QUERIES_DAYS = config['query_days']
set_analyst_theme()

In [ ]:
df_summary = run_kql(client, load_kql('30_slow_queries.kql'), days=QUERIES_DAYS, name='Slow queries summary', config=config)
df_summary['GroupLabel'] = df_summary['QueryType'].astype(str) + ' | ' + df_summary['LegalEntity'].astype(str) + ' | ' + df_summary['ExecutionMode'].astype(str)
display(df_summary.head(20))
if not df_summary.empty:
    display(metric_snapshot({
        'Grupos con slow queries': len(df_summary),
        'Max slow queries en un grupo': int(pd.to_numeric(df_summary['SlowQueries'], errors='coerce').max()),
        'Max P95 (s)': round(pd.to_numeric(df_summary['P95DurationSeconds'], errors='coerce').max(), 2),
        'Max duracion (s)': round(pd.to_numeric(df_summary['MaxDurationSeconds'], errors='coerce').max(), 2),
    }))
plot_bar(df_summary, 'GroupLabel', 'SlowQueries', 'Volumen de slow queries por grupo', top_n=15)

In [ ]:
df_timeseries = run_kql(client, load_kql('32_slow_queries_timeseries.kql'), days=QUERIES_DAYS, name='Slow queries timeseries', config=config)
display(df_timeseries.tail(24))
plot_timeseries(df_timeseries, 'timestamp', ['SlowQueries', 'P95DurationSeconds'], 'Evolucion horaria de slow queries')

df_details = run_kql(client, load_kql('31_slow_queries_details.kql'), days=QUERIES_DAYS, name='Slow queries details', config=config)
display(df_details.head(30))

## Como decidir si va a dashboard

- Un ranking por volumen ayuda a priorizar focos de optimizacion.
- La serie temporal es util cuando hay picos asociados a procesos, cierres o ventanas batch.
- La tabla de detalle no suele ir a portada, pero si como drill-through para investigacion.